# Pilot Research Analysis

Exploratory analysis of extracted sensory data:
- Descriptive statistics across papers
- Attribute vocabulary coverage
- Cross-study comparisons
- Dose-response curve visualization
- Taste profile visualization

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT_DIR = Path("..").resolve()
EXTRACTIONS_DIR = ROOT_DIR / "data" / "extractions"
DB_PATH = ROOT_DIR / "data" / "sensory_index.db"
VOCAB_PATH = ROOT_DIR / "vocabulary" / "attribute_map.json"

In [ ]:
# Load all extracted papers
papers = {}
for json_path in sorted(EXTRACTIONS_DIR.glob("*.json")):
    with open(json_path) as f:
        papers[json_path.stem] = json.load(f)

print(f"Loaded {len(papers)} papers")

## 1. Descriptive Statistics

In [ ]:
desc_data = []
for sid, paper in papers.items():
    meta = paper.get("study_metadata", {})
    exps = paper.get("experiments", [])
    total_stimuli = sum(len(e.get("stimuli", [])) for e in exps)
    max_panel = max((e.get("panel", {}).get("panel_size", 0) for e in exps), default=0)
    scales = list(set(e.get("scale", {}).get("type", "unknown") for e in exps))
    
    desc_data.append({
        "study_id": sid,
        "year": meta.get("year"),
        "journal": meta.get("journal"),
        "country": meta.get("country"),
        "food_category": meta.get("food_category"),
        "num_experiments": len(exps),
        "total_stimuli": total_stimuli,
        "panel_size": max_panel,
        "scales": ", ".join(scales),
    })

df_desc = pd.DataFrame(desc_data)
if not df_desc.empty:
    display(df_desc)
    
    print(f"\nSummary:")
    print(f"  Total experiments: {df_desc['num_experiments'].sum()}")
    print(f"  Total stimuli: {df_desc['total_stimuli'].sum()}")
    print(f"  Avg panel size: {df_desc['panel_size'].mean():.0f}")
    print(f"  Countries: {df_desc['country'].nunique()}")
    print(f"  Journals: {df_desc['journal'].nunique()}")
    print(f"  Scale types: {df_desc['scales'].unique()}")

## 2. Attribute Vocabulary Coverage

In [ ]:
with open(VOCAB_PATH) as f:
    vocab = json.load(f)

mappings = vocab.get("mappings", {})
categories = vocab.get("categories", {})

print(f"Vocabulary size: {len(mappings)} raw terms → {len(set(mappings.values()))} normalized terms")
print(f"\nCategories:")
for cat, terms in categories.items():
    print(f"  {cat}: {len(terms)} terms — {', '.join(terms)}")

unmapped = vocab.get("unmapped", [])
if unmapped:
    print(f"\nUnmapped terms ({len(unmapped)}): {unmapped}")

## 3. Cross-Study Sweetness Comparison

Compare sweetness-related data across papers (where available).

In [ ]:
# Extract sweetness data from all papers
sweetness_data = []

for sid, paper in papers.items():
    for exp in paper.get("experiments", []):
        sensory = exp.get("sensory_data", {})
        derived = exp.get("derived_metrics", {})
        scale_type = exp.get("scale", {}).get("type", "unknown")
        
        # Look for sweetness potency or dose-response data
        if isinstance(derived, dict):
            for key, val in derived.items():
                if "sweet" in key.lower() or "potency" in key.lower():
                    sweetness_data.append({
                        "study_id": sid,
                        "experiment_id": exp.get("experiment_id"),
                        "metric": key,
                        "value": val,
                        "scale": scale_type,
                    })

if sweetness_data:
    df_sweet = pd.DataFrame(sweetness_data)
    print(f"Found {len(df_sweet)} sweetness-related metrics")
    display(df_sweet)
else:
    print("No cross-study sweetness data found yet.")
    print("This analysis becomes possible once multiple papers are extracted.")

## 4. Dose-Response Curve Visualization

Plot extracted dose-response data from papers that include it.

In [ ]:
def extract_dose_response(paper):
    """Extract dose-response curves from a paper's sensory data."""
    curves = []
    for exp in paper.get("experiments", []):
        sensory = exp.get("sensory_data", {})
        if not isinstance(sensory, dict):
            continue
        for key, val in sensory.items():
            if isinstance(val, dict) and "concentrations" in val and "means" in val:
                curves.append({
                    "compound": key,
                    "concentrations": val["concentrations"],
                    "means": val["means"],
                    "sems": val.get("sems"),
                    "unit": val.get("unit", "M"),
                    "scale": val.get("scale", "unknown"),
                })
    return curves

all_curves = []
for sid, paper in papers.items():
    curves = extract_dose_response(paper)
    for c in curves:
        c["study_id"] = sid
    all_curves.extend(curves)

if all_curves:
    fig, ax = plt.subplots(figsize=(12, 7))
    for curve in all_curves[:20]:  # Plot up to 20 curves
        label = f"{curve['study_id']} — {curve['compound']}"
        ax.plot(curve["concentrations"], curve["means"], marker="o", label=label)
        if curve.get("sems"):
            ax.fill_between(
                curve["concentrations"],
                [m - s for m, s in zip(curve["means"], curve["sems"])],
                [m + s for m, s in zip(curve["means"], curve["sems"])],
                alpha=0.15,
            )
    ax.set_xlabel(f"Concentration ({all_curves[0].get('unit', 'M')})")
    ax.set_ylabel("Sweetness Intensity")
    ax.set_title("Dose-Response Curves Across Studies")
    ax.set_xscale("log")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No dose-response data extracted yet.")

## 5. Taste Profile Comparison

Radar/spider plots comparing taste profiles across papers.

In [ ]:
def make_radar_plot(profiles, title="Taste Profiles"):
    """Create a radar plot from taste profile data."""
    if not profiles:
        print("No profile data to plot.")
        return
    
    # Get all attributes
    all_attrs = set()
    for p in profiles:
        all_attrs.update(p["attributes"].keys())
    attrs = sorted(all_attrs)
    N = len(attrs)
    if N < 3:
        print("Need at least 3 attributes for radar plot.")
        return
    
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]  # Close the polygon
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    for p in profiles:
        values = [p["attributes"].get(a, 0) for a in attrs]
        values += values[:1]
        ax.plot(angles, values, "o-", label=p["label"])
        ax.fill(angles, values, alpha=0.1)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(attrs)
    ax.set_title(title)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()

# Extract taste profiles from papers
profiles = []
for sid, paper in papers.items():
    for exp in paper.get("experiments", []):
        sensory = exp.get("sensory_data", {})
        if isinstance(sensory, dict) and "by_stimulus" in sensory:
            for stim_name, stim_data in sensory["by_stimulus"].items():
                if isinstance(stim_data, dict):
                    attrs = {}
                    for attr, val in stim_data.items():
                        if isinstance(val, dict) and "mean" in val:
                            attrs[attr] = val["mean"]
                    if attrs:
                        profiles.append({
                            "label": f"{sid}: {stim_name}",
                            "attributes": attrs,
                        })

if profiles:
    make_radar_plot(profiles[:8], "Taste Profiles Across Studies")
else:
    print("No taste profile data extracted yet.")
    print("This visualization becomes available once papers with attribute ratings are processed.")